<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大语言模型》（Build a Large Language Model From Scratch）</a> 一书的配套代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 使用 GPT-4 进行反思微调（Reflection Tuning）


- 本 notebook 使用 OpenAI 的 GPT-4 API 实现 [Reflection-Tuning: Data Recycling Improves LLM Instruction-Tuning](https://arxiv.org/abs/2310.11716) 论文中的数据集改进流程

![](https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/reflection-tuning/reflection-tuning.webp)

- 在原论文中，研究人员改进了 [Alpaca](https://huggingface.co/datasets/tatsu-lab/alpaca) 和 [WizardLM](https://huggingface.co/datasets/WizardLMTeam/WizardLM_evol_instruct_70k) 指令微调数据集；在本 notebook 中，我们改进 [第 7 章使用的 instruction 数据集](../01_main-chapter-code/ch07_ch.ipynb)（[instruction-data.json](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/01_main-chapter-code/instruction-data.json)）（不过由于格式与 Alpaca 相同，同样的代码也适用于 Alpaca 数据集）

- 预期数据集格式如下：

```python
    {
        "instruction": "Edit the following sentence for grammar.",
        "input": "He go to the park every day.",
        "output": "He goes to the park every day."
    },
    {
        "instruction": "Convert 45 kilometers to meters.",
        "input": "",
        "output": "45 kilometers is 45000 meters."
    },
```

> 请注意，本 notebook 复现了论文中的方法，作者使用 GPT API 增强现有数据集。然而需要了解的是，根据 [OpenAI 使用条款](https://openai.com/policies/row-terms-of-use/) 的规定，GPT API 生成的数据可能不得用于开发与 OpenAI 竞争的模型："What you cannot do... Use Output to develop models that compete with OpenAI."
相关讨论见 [此处](https://www.reddit.com/r/LocalLLaMA/comments/17vbg1f/does_openai_tos_prohibit_generating_datasets_for/))。

In [ ]:
# pip install -r requirements-extra.txt

In [ ]:
from importlib.metadata import version

pkgs = [
    "openai",  # OpenAI API
    "tqdm",    # 进度条
]

for p in pkgs:
    print(f"{p} 版本: {version(p)}")

## 测试 OpenAI API


- 首先，让我们测试 OpenAI API 是否正确配置
- 如果还没有账户，需要在 https://platform.openai.com/ 注册
- 注意：还需要向账户充值，因为 GPT-4 API 并非免费（参见 https://platform.openai.com/settings/organization/billing/overview）
- 按本 notebook 原样运行代码，使用 GPT-4o-mini 时费用约为 \$0.03（3 美分）
- 将上述两种方法应用于第 7 章 instruction 数据集的全部 1100 条条目，费用约为 \$0.60（60 美分）

- 首先，我们需要提供 OpenAI API 密钥，可在 https://platform.openai.com/api-keys 获取
- 切勿与任何人分享此密钥
- 将此密钥（`"sk-..."`）添加到本文件夹的 `config.json` 文件中

In [ ]:
import json
from openai import OpenAI

# 从 JSON 文件加载 API 密钥。
# 确保将 "sk-..." 替换为你在 https://platform.openai.com/api-keys 获取的实际 API 密钥
with open("config.json", "r") as config_file:
    config = json.load(config_file)
    api_key = config["OPENAI_API_KEY"]

client = OpenAI(api_key=api_key)

- 首先，让我们用一个简单示例测试 API，确保其按预期工作：


In [ ]:
def run_chatgpt(prompt, client, model="gpt-4o-mini", system_prompt=None):
    # 如果提供了 system_prompt，则定义系统消息
    messages = []
    
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    
    # 将用户 prompt 添加到消息中
    messages.append({"role": "user", "content": prompt})

    # 调用 API
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.0,
        seed=123,
    )
    
    # 返回模型响应
    return response.choices[0].message.content


prompt = "Respond with 'hello world' if you got this message."
run_chatgpt(prompt, client)

## 加载 JSON 条目


- 接下来，加载并处理 instruction 数据集
- 此处假设我们将测试数据集和模型响应保存为如下 JSON 文件：

In [ ]:
from pathlib import Path


json_file = Path("..") / "01_main-chapter-code" / "instruction-data.json"

with open(json_file, "r") as file:
    json_data = json.load(file)

print("条目数量:", len(json_data))

- 打印一条数据集条目以查看其结构：


In [ ]:
from pprint import pp as pprint

pprint(json_data[0])

## 改进指令


- Reflection-Tuning 作者分享了两种方法：(1) 改进指令 (2) 改进回复
- 让我们从改进数据集中的指令开始
- 以下是来自 [Reflection-Tuning 仓库](https://github.com/tianyi-lab/Reflection_Tuning/blob/main/reflection_code/reflect_response.py) 的小工具函数，用于为数据集改进格式化 GPT-4 模型的输入

In [ ]:
def build_instruction_reflection_prompt_no_input(ins, outp):

    sys_prompt = "You are a helpful, precise but picky assistant for checking the quality of a given instruction."
    prompt_template = "[Instruction]\n{ins}\n\n[The Start of Answer]\n{outp}\n\n[The End of Answer]\n\n[System]\n{criteria}\n\n"
    criteria = "We would like you to answer several questions related to the quality of a given instruction. \n" + \
                "1. Why this instruction is not good? First analyse the instruction based on Complexity of the Topic, Level of Detail Required, Knowledge Required, Ambiguity of the Instruction and Logical Reasoning or Problem-Solving Involved. \n" + \
                "Then analyse why this answer is not good for the given instruction? Analyse based on the Helpfulness, Relevance, Accuracy and Level of Details. \n" + \
                "Finally analyse why this bad instruction lead to a bad answer. " +\
                "2. Based on the reason you provided, generate a new and complete instruction which is complex and difficult to answer directly. " + \
                "Make sure the new instruction is relevent but independent to the original instruction, which can be answered without knowing the original instruction, put the new instruction in the format of [New Instruction] your instruction [End]" +\
                "3. Answer the newly generated instruction as detailed as possible, in the format of [New Answer] your answer [End] \n"
    prompt = prompt_template.format(
        ins=ins, outp=outp, criteria=criteria
    )
    return sys_prompt, prompt

- 为演示其工作原理，考虑数据集条目 `json_data[2]`


In [ ]:
pprint(json_data[2])

- 我们可以使用上文定义的 `build_instruction_reflection_prompt_no_input` 函数改进指令，如下所示：


In [ ]:
entry = json_data[2]

system_prompt, prompt = build_instruction_reflection_prompt_no_input(ins=entry["instruction"], outp=entry["output"])
output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)

print(output)

- 响应非常冗长，这对分析很有用；此外，它还有助于 GPT-4 模型通过思维链（chain-of-thought）提示方法进行改进
- 然而，为构建改进后的数据集，我们实际上只关心新指令和新输出，而非分析内容
- 我们可以使用 [Reflection-Tuning 仓库](https://github.com/tianyi-lab/Reflection_Tuning/blob/main/reflection_code/reflect_response.py) 中的以下工具代码，从 GPT-4 输出中提取改进后的指令和输出

In [ ]:
import re

def extract_instruction_segment(text, no_input=True):
    if '[New Instruction]' in text:
        pattern = r'(\[New Instruction\])(.*?)(\[End\]|\[New Answer\]|New Answer:)'
    else:
        pattern = r'(New Instruction:)(.*?)(\[End\]|\[New Answer\]|New Answer:)'
    segments = re.findall(pattern, text, re.DOTALL)
    if len(segments) == 0:
        seg_ins = ''
    else:
        seg_ins = segments[0][1].strip()
    if seg_ins.endswith("\n\n3."):
        seg_ins = seg_ins[:-4]
    return seg_ins


def extract_output_segment(text, no_input=True):
    if '[New Answer]' in text:
        pattern = r'(\[New Answer\])(.*?)(\[End\]|$)'
    else:
        pattern = r'(New Answer:)(.*?)(\[End\]|$)'
        # pattern = r'(\[New Answer\]|New Answer:)(.*?)(\[End\]|$)'
    segments = re.findall(pattern, text, re.DOTALL)
    if len(segments) == 0:
        seg_oup = ''
    else:
        seg_oup = segments[0][1].strip()
    return seg_oup


def extract_instruction(text):
    if text == '':
        return []
    seg_ins = extract_instruction_segment(text, no_input=True)
    seg_oup = extract_output_segment(text, no_input=True)
    return [seg_ins, seg_oup]

- 让我们使用这些工具函数，从之前生成的冗长 GPT-4 输出中提取改进后的指令和回复：


In [ ]:
new_instr, new_outp = extract_instruction(output)

In [ ]:
print(new_instr)

In [ ]:
print(new_outp)

- 注意，当前指令改进仅针对没有 `"input"` 字段的数据集条目实现


## 改进回复


- 类似地，我们也可以将 Reflection-Tuning 改进流程专门应用于数据集的回复（即 "output" 字段）
- 以下是来自 [Reflection-Tuning 仓库](https://github.com/tianyi-lab/Reflection_Tuning/blob/main/reflection_code/reflect_response.py) 的两个小工具函数，用于为数据集改进格式化 GPT-4 模型的输入

In [ ]:
def build_response_reflection_prompt_no_input(ins, outp):

    sys_prompt = "You are a helpful, precise but picky assistant for checking the quality of the answer to a given instruction."
    prompt_template = "[Instruction]\n{ins}\n\n[The Start of Answer]\n{outp}\n\n[The End of Answer]\n\n[System]\n{criteria}\n\n"
    criteria = "We would like you to answer several questions related to the quality of the answer to the given instruction. \n" + \
                "1. Why this answer is not good for the given instruction? Analyse based on the Helpfulness, Relevance, Accuracy and Level of Details. \n" + \
                "2. Based on the reason you provided, generate a better answer, new and complete, as detailed as possible, in the format of [Better Answer] your answer [End] \n" 
    prompt = prompt_template.format(
        ins=ins, outp=outp, criteria=criteria
    )
    return sys_prompt, prompt


def build_response_reflection_prompt_with_input(ins, inp, outp):

    sys_prompt = "You are a helpful and precise assistant for checking the quality of the answer to a given instruction and its input."
    prompt_template = "[Instruction]\n{ins}\n\n[The Start of Input]\n{inp}\n\n[The End of Input]\n\n[The Start of Answer]\n{outp}\n\n[The End of Answer]\n\n[System]\n{criteria}\n\n"
    criteria = "We would like you to answer several questions related to the quality of the answer to the given instruction and corresponding input. \n" + \
                "1. Why this answer is not good for the given instruction and corresponding input? Analyse based on the Helpfulness, Relevance, Accuracy and Level of Details. \n" + \
                "2. Based on the reason you provided, generate a better answer, new and complete, as detailed as possible, in the format of [Better Answer] your answer [End] \n" 
    prompt = prompt_template.format(
        ins=ins, inp=inp, outp=outp, criteria=criteria
    )
    return sys_prompt, prompt

- 再次对一条数据集条目进行应用，查看其工作原理并生成改进后的回复：


In [ ]:
entry = json_data[2]

system_prompt, prompt = build_response_reflection_prompt_no_input(ins=entry["instruction"], outp=entry["output"])
output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)

print(output)

- 如上所示，响应包含对原始回复的分析；我们可以使用 [Reflection-Tuning 仓库](https://github.com/tianyi-lab/Reflection_Tuning/blob/main/reflection_code/reflect_response.py) 中的以下工具函数提取新回复

In [ ]:
def extract_response(text):
    if text.count('[Better Answer]') >= 2:
        pattern = r'\[(Better Answer)\](.*?)(\[End\]|\[Better Answer\]|$)'
        segments = re.findall(pattern, text, re.DOTALL)
    else:
        # pattern = r'\[(Better Answer)\](.*?)\[End\]'
        pattern = r'\[(Better Answer)\](.*?)(\[End\]|End|$)'
        segments = re.findall(pattern, text, re.DOTALL)
    return [segment[1].strip() for segment in segments]

In [ ]:
response = extract_response(output)[0]
print(response)

## 改进数据集


- 现在，让我们将指令反思和回复反思技术应用于实际数据集
- 注意：此处仅为演示目的应用于一小部分数据；要对整个数据集应用，请将

```python
data_to_process = json_data[:3]
```

改为

```python
data_to_process = json_data
```

### 反思指令


- 以下代码将 Reflection-Tuning 数据集改进方法应用于原始数据集中的指令


In [ ]:
data_to_process = json_data[:3]

In [ ]:
from tqdm import tqdm


def reflect_instructions(json_data, client):
    new_json_data = [] 
    
    for entry in tqdm(json_data):
        
        if not entry["input"]:
            system_prompt, prompt = build_instruction_reflection_prompt_no_input(ins=entry["instruction"], outp=entry["output"])
            output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)
            new_instr, new_outp = extract_instruction(output)
            new_entry = {"instruction": new_instr, "input": "", "output": new_outp}
            new_json_data.append(new_entry)
        else:
            new_json_data.append(entry)

    return new_json_data

In [ ]:
data_to_process = json_data[:3]

new_json_data = reflect_instructions(data_to_process, client)

In [ ]:
for i in new_json_data[:3]:
    pprint(i)
    print("\n\n")

- 保存新数据集：


In [ ]:
with open("instruction-reflected.json", "w") as file:
    json.dump(new_json_data, file, indent=4)

### 反思回复


- 现在对回复反思执行相同操作：


In [ ]:
data_to_process = json_data[:3]

In [ ]:
def reflect_responses(json_data, client):
    new_json_data = [] 
    
    for entry in tqdm(json_data):
        
        if not entry["input"]:
            system_prompt, prompt = build_response_reflection_prompt_no_input(ins=entry["instruction"], outp=entry["output"])
            output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)
            new_response = extract_response(output)

            if not len(new_response):
                new_response = entry["output"]
                      
            new_entry = {"instruction": entry["instruction"], "input": "", "output": new_response[0]}
            new_json_data.append(new_entry)

        else:
            system_prompt, prompt = build_response_reflection_prompt_with_input(ins=entry["instruction"], inp=entry["input"], outp=entry["output"])
            output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)
            new_response = extract_response(output)

            if not len(new_response):
                new_response = entry["output"]

            new_entry = {"instruction": entry["instruction"], "input": entry["input"], "output": new_response[0]}
            new_json_data.append(new_entry)

    return new_json_data

In [ ]:
new_json_data = reflect_responses(data_to_process, client)

In [ ]:
for i in new_json_data[:3]:
    pprint(i)
    print("\n\n")

- 保存新数据集：


In [ ]:
with open("response-reflected.json", "w") as file:
    json.dump(new_json_data, file, indent=4)

## 创建改进后的指令数据


- 将上述两种方法应用于第 7 章 instruction 数据集的全部 1100 条条目，费用约为 \$0.60（60 美分）
- 为避免 GitHub 仓库因数据集文件而过于臃肿，生成的数据集文件可从 Google Drive 获取：
  - [instruction-reflected.json](https://drive.google.com/file/d/1c1QnuTdt9nP1u51vBn4_b05mWR_ZNGBv/view?usp=sharing)
  - [response-reflected.json](https://drive.google.com/file/d/1RNckTZ2ELcdUoJtaylao6NvyZPMtNv1v/view?usp=sharing)